# 8 よく利用される売買戦略

In [7]:
import os
import sys
from dotenv import load_dotenv
from pathlib import Path

import plotly.graph_objs as go
import talib as ta
import datetime as dt
import pandas as pd
import numpy as np

In [2]:
sys.path.append("/home/jovyan/notebook")
from Modules.get_market_data import GetMarketData
get_market_data = GetMarketData(Path('/home/jovyan/data'))

### 8.1.1 グランビルの法則

### 8.1.2 チャートで確認

#### 指数移動平均線（EMA: Exponential Moving Average）

EMA は過去の価格に対して**指数的に減衰する重みを付けた移動平均**で、単純移動平均（SMA）より直近の価格変動を強調する。

$$
EMA_t = \alpha \cdot P_t + (1 - \alpha) \cdot EMA_{t-1}, \quad \alpha = \frac{2}{N + 1}
$$

| 変数 | 意味 |
|------|------|
| $P_t$ | $t$ 日の終値 |
| $N$ | 期間（本例では 200 日）|
| $\alpha$ | 平滑化係数（$0 < \alpha \leq 1$）|

- $EMA_{200}$ が株価の**長期トレンド基準線**として機能する。
- 株価が $EMA_{200}$ を上回ると強気、下回ると弱気トレンドと判断される。
- SMA との比較: $\alpha = \frac{1}{N}$ とすれば SMA に近似するが、EMA の方が最新値への反応が早い。

In [4]:
name = "日本製鉄"
code = "5401.T"

# チャート表示開始日
display_start_date = dt.datetime(2020, 1, 1)
# データ取得終了日(現在の日付を取得)
end_date = dt.datetime(2022, 3, 31)
# データ取得開始日
start_date = display_start_date - dt.timedelta(days=300)

### 日本製鉄 (5401.T)
df = get_market_data.get_data_from_yfinance(code, start=start_date.strftime('%Y-%m-%d'), end=end_date.strftime('%Y-%m-%d'))
if isinstance(df.columns, pd.MultiIndex):
    df = df.droplevel(level=1, axis=1)
close = df['Close']

# 移動平均線
#df["ma200"] = ta.SMA(close, timeperiod=200)
df["ma200"] = ta.EMA(df["Close"], timeperiod=200)

rdf = df[display_start_date:end_date]

# インデクスを文字列型に
rdf.index = pd.to_datetime(rdf.index).strftime('%m-%d-%Y')

# 200日線と終値を折れ線グラフで表示
layout = {"height": 600,
          "title": { "text": " {} {} ".format(code, name), "x": 0.5 },
          "xaxis": { "rangeslider": {"visible": False} },
          "yaxis": { "title": "株価 (円)", "side": "left", "tickformat": "," },
          "plot_bgcolor": "light blue",
}

data = [
    go.Scatter(x=rdf.index, y=rdf["ma200"], mode="lines", name="MA200",
               line={ "color": "orange", "width": 2 }),
    go.Scatter(x=rdf.index, y=rdf["Close"], mode="lines", name="Close",
               line={ "color": "green", "width": 2 }),
]

fig = go.Figure(layout=go.Layout(layout), data=data)

# X軸日付調整
fig.update_layout({
    "xaxis": {
        "ticktext": [x.split("-")[0] + "/" + x.split("-")[1] for x in rdf.index[::20]],
        "tickvals": [x for x in rdf.index[::20]],
    }
})

f = go.Figure(fig)
f.show()

[*********************100%***********************]  1 of 1 completed


## 8.2 ローソク足パターン
### 8.2.1 ローソク足パターンとは

#### ローソク足の基本構成

1本のローソク足は**4本値**（始値 $O$, 高値 $H$, 安値 $L$, 終値 $C$）で構成される。

$$
\text{実体} = |C_t - O_t|, \quad
\text{上ヒゲ} = H_t - \max(O_t,\, C_t), \quad
\text{下ヒゲ} = \min(O_t,\, C_t) - L_t
$$

| 条件 | 足の種類 |
|------|----------|
| $C_t > O_t$ | 陽線（上昇） |
| $C_t < O_t$ | 陰線（下落） |
| $C_t = O_t$ | 同時線 |

TA-Lib の各パターン検出関数は、これらの4本値の**大小関係と比率**をルールとして定義しシグナルを返す。

In [10]:
name = "積水ハウス"
code = "1928.T"

# チャート表示開始日
display_start_date = dt.datetime(2021, 11, 1)
# データ取得終了日(現在の日付を取得)
end_date = dt.datetime(2022, 3, 31)
# データ取得開始日
start_date = display_start_date - dt.timedelta(days=300)

### 積水ハウス (1928.T)
df = get_market_data.get_data_from_yfinance(code, start=start_date.strftime('%Y-%m-%d'), end=end_date.strftime('%Y-%m-%d'))
if isinstance(df.columns, pd.MultiIndex):
    df = df.droplevel(level=1, axis=1)
close = df['Close']

layout = {"height": 600,
          "title": { "text": " {} {} ".format(code, name), "x": 0.5 },
          "xaxis": { "rangeslider": {"visible": False} },
          "yaxis": { "title": "株価 (円)", "side": "left", "tickformat": "," },
          "plot_bgcolor": "light blue",
}

rdf = df[display_start_date:end_date]
# インデクスを文字列型に
rdf.index = pd.to_datetime(rdf.index).strftime('%m-%d-%Y')

# ローソク足
data = [go.Candlestick(x=rdf.index,
                    open=rdf['Open'], high=rdf['High'],
                    low=rdf['Low'], close=rdf['Close'],
                    increasing_line_color='red', 
                    decreasing_line_color='gray',),
]

fig = go.Figure(layout=go.Layout(layout), data=data)
fig["layout"].update({
    "xaxis": {
        "tickvals": rdf.index[::2],
        "ticktext": [" {} {} ".format(x.split("-")[0], x.split("-")[1]) for x in rdf.index[::2]],
    }
})

fig.show()

[*********************100%***********************]  1 of 1 completed


### 8.2.2 TA-Libでローソク足パターンを検出する

#### 丸坊主（CDLMARUBOZU）— 1本足パターン

上ヒゲ・下ヒゲが完全にない（または極めて小さい）ローソク足。実体が高値から安値までを全て占める。

$$
\text{陽の丸坊主}: \quad O_t = L_t \;\text{ かつ }\; C_t = H_t, \quad C_t > O_t \quad (+100)
$$

$$
\text{陰の丸坊主}: \quad O_t = H_t \;\text{ かつ }\; C_t = L_t, \quad C_t < O_t \quad (-100)
$$

- **陽の丸坊主**: 買い圧力が終日継続した強気シグナル（寄り付きが安値、引けが高値）
- **陰の丸坊主**: 売り圧力が終日継続した弱気シグナル（寄り付きが高値、引けが安値）
- ヒゲが全くないため、相場参加者の方向性が完全に一致していることを示す

In [11]:
name = "積水ハウス"
code = "1928.T"

# チャート表示開始日
display_start_date = dt.datetime(2021, 11, 1)
# データ取得終了日(現在の日付を取得)
end_date = dt.datetime(2022, 3, 31)
# データ取得開始日
start_date = display_start_date - dt.timedelta(days=300)

### 積水ハウス (1928.T)
df = get_market_data.get_data_from_yfinance(code, start=start_date.strftime('%Y-%m-%d'), end=end_date.strftime('%Y-%m-%d'))
if isinstance(df.columns, pd.MultiIndex):
    df = df.droplevel(level=1, axis=1)
close = df['Close']

layout = {"height": 600,
          "title": { "text": " {} {} ".format(code, name), "x": 0.5 },
          "xaxis": { "rangeslider": {"visible": False} },
          "yaxis": { "title": "株価 (円)", "side": "left", "tickformat": "," },
          "plot_bgcolor": "light blue",
}

marubouzu = ta.CDLMARUBOZU(df['Open'], df['High'], df['Low'], df['Close'])
# チャートに表示するテキスト
df["marubouzu_text"] = marubouzu.replace({100: "買い", -100: "売り", 0: ""})
# チャートに表示するマーカーの位置を算出
df["marubouzu_marker"] = (marubouzu / 100 * df["High"]).abs().replace({0: np.nan})

rdf = df[display_start_date:end_date]
# インデクスを文字列型に
rdf.index = pd.to_datetime(rdf.index).strftime('%m-%d-%Y')

# ローソク足
data = [go.Candlestick(x=rdf.index,
                    open=rdf['Open'], high=rdf['High'],
                    low=rdf['Low'], close=rdf['Close'],
                    increasing_line_color='red', 
                    decreasing_line_color='gray',),
        # 検出パターン
        go.Scatter(x=rdf.index,
                y=rdf["marubouzu_marker"],
                mode="markers+text",
                text=rdf["marubouzu_text"],
                textposition="top center",
                name="丸坊主",
                marker={ "size": 12, "color": "blue",
                        "opacity": 0.6 },
                textfont={ "size": 14, "color": "black" }),
]

fig = go.Figure(layout=go.Layout(layout), data=data)
fig["layout"].update({
    "xaxis": {
        "tickvals": rdf.index[::2],
        "ticktext": [" {} {} ".format(x.split("-")[0], x.split("-")[1]) for x in rdf.index[::2]],
    }
})

fig.show()

[*********************100%***********************]  1 of 1 completed


#### 寄付坊主 / 大引坊主（CDLBELTHOLD）— 1本足パターン

片側のヒゲがなく、始値または終値が高値・安値の一端と一致するローソク足。丸坊主より条件が緩い。

$$
\text{寄付坊主（陽線）}: \quad O_t = L_t \;\text{ かつ }\; C_t > O_t \quad (+100)
$$

$$
\text{大引坊主（陰線）}: \quad O_t = H_t \;\text{ かつ }\; C_t < O_t \quad (-100)
$$

| パターン | 条件 | 意味 |
|----------|------|------|
| 寄付坊主（陽） | 寄り付きが安値（$O_t = L_t$）、上ヒゲはあってもよい | 買い圧力の強さ |
| 大引坊主（陰） | 寄り付きが高値（$O_t = H_t$）、下ヒゲはあってもよい | 売り圧力の強さ |

**丸坊主との違い**: 反対側のヒゲ（終値側）は存在してよい（$C_t \neq H_t$ または $C_t \neq L_t$ は許容）。

In [12]:
name = "積水ハウス"
code = "1928.T"

# チャート表示開始日
display_start_date = dt.datetime(2021, 11, 1)
# データ取得終了日(現在の日付を取得)
end_date = dt.datetime(2022, 3, 31)
# データ取得開始日
start_date = display_start_date - dt.timedelta(days=300)

### 積水ハウス (1928.T)
df = get_market_data.get_data_from_yfinance(code, start=start_date.strftime('%Y-%m-%d'), end=end_date.strftime('%Y-%m-%d'))
if isinstance(df.columns, pd.MultiIndex):
    df = df.droplevel(level=1, axis=1)
close = df['Close']

layout = {"height": 600,
          "title": { "text": " {} {} ".format(code, name), "x": 0.5 },
          "xaxis": { "rangeslider": {"visible": False} },
          "yaxis": { "title": "株価 (円)", "side": "left", "tickformat": "," },
          "plot_bgcolor": "light blue",
}

belthold = ta.CDLBELTHOLD(df['Open'], df['High'], df['Low'], df['Close'])
# チャートに表示するテキスト
df["belthold_text"] = belthold.replace({100: "買い", -100: "売り", 0: ""})
# チャートに表示するマーカーの位置を算出
df["belthold_marker"] = (belthold / 100 * df["High"]).abs().replace({0: np.nan})

rdf = df[display_start_date:end_date]
# インデクスを文字列型に
rdf.index = pd.to_datetime(rdf.index).strftime('%m-%d-%Y')

# ローソク足
data = [go.Candlestick(x=rdf.index,
                    open=rdf['Open'], high=rdf['High'],
                    low=rdf['Low'], close=rdf['Close'],
                    increasing_line_color='red', 
                    decreasing_line_color='gray',),
        # 検出パターン
        go.Scatter(x=rdf.index,
                y=rdf["belthold_marker"],
                mode="markers+text",
                text=rdf["belthold_text"],
                textposition="top center",
                name="寄付坊主 / 大引坊主",
                marker={ "size": 12, "color": "blue",
                        "opacity": 0.6 },
                textfont={ "size": 14, "color": "black" }),
]

fig = go.Figure(layout=go.Layout(layout), data=data)
fig["layout"].update({
    "xaxis": {
        "tickvals": rdf.index[::2],
        "ticktext": [" {} {} ".format(x.split("-")[0], x.split("-")[1]) for x in rdf.index[::2]],
    }
})

fig.show()

[*********************100%***********************]  1 of 1 completed


### 8.2.4 2本足のローソク足パターン

#### 包み足（CDLENGULFING）— 2本足パターン

本日の実体が前日の実体全体を「包む」（上回りかつ下回る）逆張りの反転シグナル。

$$
\text{陽の包み足（強気反転）}: \quad O_{t-1} > C_{t-1} \;\text{（前日陰線）} \;\text{ かつ }\; O_t \leq C_{t-1} \;\text{ かつ }\; C_t \geq O_{t-1} \quad (+100)
$$

$$
\text{陰の包み足（弱気反転）}: \quad O_{t-1} < C_{t-1} \;\text{（前日陽線）} \;\text{ かつ }\; O_t \geq C_{t-1} \;\text{ かつ }\; C_t \leq O_{t-1} \quad (-100)
$$

| 変数 | 意味 |
|------|------|
| $O_t,\; C_t$ | 本日の始値・終値 |
| $O_{t-1},\; C_{t-1}$ | 前日の始値・終値 |

**ポイント**: 本日足の「実体」が前日足の「実体」をすべて飲み込む大きさが必要。  
相場の転換点でよく現れる、代表的な2本足の反転パターン。

In [13]:
name = "積水ハウス"
code = "1928.T"

# チャート表示開始日
display_start_date = dt.datetime(2021, 11, 1)
# データ取得終了日(現在の日付を取得)
end_date = dt.datetime(2022, 3, 31)
# データ取得開始日
start_date = display_start_date - dt.timedelta(days=300)

### 積水ハウス (1928.T)
df = get_market_data.get_data_from_yfinance(code, start=start_date.strftime('%Y-%m-%d'), end=end_date.strftime('%Y-%m-%d'))
if isinstance(df.columns, pd.MultiIndex):
    df = df.droplevel(level=1, axis=1)
close = df['Close']

layout = {"height": 600,
          "title": { "text": " {} {} ".format(code, name), "x": 0.5 },
          "xaxis": { "rangeslider": {"visible": False} },
          "yaxis": { "title": "株価 (円)", "side": "left", "tickformat": "," },
          "plot_bgcolor": "light blue",
}

engulfing = ta.CDLENGULFING(df['Open'], df['High'], df['Low'], df['Close'])
# チャートに表示するテキスト
df["engulfing_text"] = engulfing.replace({100: "買い", -100: "売り", 0: ""})
# チャートに表示するマーカーの位置を算出
df["engulfing_marker"] = (engulfing / 100 * df["High"]).abs().replace({0: np.nan})

rdf = df[display_start_date:end_date]
# インデクスを文字列型に
rdf.index = pd.to_datetime(rdf.index).strftime('%m-%d-%Y')

# ローソク足
data = [go.Candlestick(x=rdf.index,
                    open=rdf['Open'], high=rdf['High'],
                    low=rdf['Low'], close=rdf['Close'],
                    increasing_line_color='red', 
                    decreasing_line_color='gray',),
        # 検出パターン
        go.Scatter(x=rdf.index,
                y=rdf["engulfing_marker"],
                mode="markers+text",
                text=rdf["engulfing_text"],
                textposition="top center",
                name="包み足",
                marker={ "size": 12, "color": "blue",
                        "opacity": 0.6 },
                textfont={ "size": 14, "color": "black" }),
]

fig = go.Figure(layout=go.Layout(layout), data=data)
fig["layout"].update({
    "xaxis": {
        "tickvals": rdf.index[::2],
        "ticktext": [" {} {} ".format(x.split("-")[0], x.split("-")[1]) for x in rdf.index[::2]],
    }
})

fig.show()

[*********************100%***********************]  1 of 1 completed


#### はらみ足（CDLHARAMI）— 2本足パターン

本日のローソク足の実体が前日の実体の**内側に完全に収まる**パターン。相場の勢いが鈍化し、反転の可能性を示す。

$$
\min(O_{t-1},\, C_{t-1}) < \min(O_t,\, C_t) \;\text{ かつ }\; \max(O_t,\, C_t) < \max(O_{t-1},\, C_{t-1})
$$

$$
\text{陽のはらみ（強気）}: \quad O_{t-1} > C_{t-1} \;\text{（前日陰線）} \;\text{ かつ }\; C_t > O_t \;\text{（本日陽線）} \quad (+100)
$$

$$
\text{陰のはらみ（弱気）}: \quad O_{t-1} < C_{t-1} \;\text{（前日陽線）} \;\text{ かつ }\; C_t < O_t \;\text{（本日陰線）} \quad (-100)
$$

**包み足との対比**:

| パターン | 本日の実体と前日の関係 |
|----------|----------------------|
| 包み足（Engulfing）| 本日が前日を「包む」（本日 ＞ 前日）|
| はらみ足（Harami） | 本日が前日に「収まる」（本日 ＜ 前日）|

In [14]:
name = "積水ハウス"
code = "1928.T"

# チャート表示開始日
display_start_date = dt.datetime(2021, 11, 1)
# データ取得終了日(現在の日付を取得)
end_date = dt.datetime(2022, 3, 31)
# データ取得開始日
start_date = display_start_date - dt.timedelta(days=300)

### 積水ハウス (1928.T)
df = get_market_data.get_data_from_yfinance(code, start=start_date.strftime('%Y-%m-%d'), end=end_date.strftime('%Y-%m-%d'))
if isinstance(df.columns, pd.MultiIndex):
    df = df.droplevel(level=1, axis=1)
close = df['Close']

layout = {"height": 600,
          "title": { "text": " {} {} ".format(code, name), "x": 0.5 },
          "xaxis": { "rangeslider": {"visible": False} },
          "yaxis": { "title": "株価 (円)", "side": "left", "tickformat": "," },
          "plot_bgcolor": "light blue",
}

harami = ta.CDLHARAMI(df['Open'], df['High'], df['Low'], df['Close'])
# チャートに表示するテキスト
df["harami_text"] = harami.replace({100: "買い", -100: "売り", 0: ""})
# チャートに表示するマーカーの位置を算出
df["harami_marker"] = (harami / 100 * df["High"]).abs().replace({0: np.nan})

rdf = df[display_start_date:end_date]
# インデクスを文字列型に
rdf.index = pd.to_datetime(rdf.index).strftime('%m-%d-%Y')

# ローソク足
data = [go.Candlestick(x=rdf.index,
                    open=rdf['Open'], high=rdf['High'],
                    low=rdf['Low'], close=rdf['Close'],
                    increasing_line_color='red', 
                    decreasing_line_color='gray',),
        # 検出パターン
        go.Scatter(x=rdf.index,
                y=rdf["harami_marker"],
                mode="markers+text",
                text=rdf["harami_text"],
                textposition="top center",
                name="はらみ足",
                marker={ "size": 12, "color": "blue",
                        "opacity": 0.6 },
                textfont={ "size": 14, "color": "black" }),
]

fig = go.Figure(layout=go.Layout(layout), data=data)
fig["layout"].update({
    "xaxis": {
        "tickvals": rdf.index[::2],
        "ticktext": [" {} {} ".format(x.split("-")[0], x.split("-")[1]) for x in rdf.index[::2]],
    }
})

fig.show()

[*********************100%***********************]  1 of 1 completed


### 8.2.5 3本のローソク足パターン

#### 包み上げ / 包み下げ（CDL3OUTSIDE）— 3本足パターン

包み足（2本目で反転）の後、3本目のローソク足でその方向性を**確定**させる強力な反転シグナル。

$$
\text{包み上げ（強気反転）}:
\begin{cases}
\text{Day1}: & O_{t-2} > C_{t-2} \quad \text{（陰線）} \\
\text{Day2}: & O_{t-1} \leq C_{t-2} \;\text{ かつ }\; C_{t-1} \geq O_{t-2} \quad \text{（Day1を包む陽線）} \\
\text{Day3}: & C_t > C_{t-1} \quad \text{（上方確定）}
\end{cases}
\quad (+100)
$$

$$
\text{包み下げ（弱気反転）}:
\begin{cases}
\text{Day1}: & O_{t-2} < C_{t-2} \quad \text{（陽線）} \\
\text{Day2}: & O_{t-1} \geq C_{t-2} \;\text{ かつ }\; C_{t-1} \leq O_{t-2} \quad \text{（Day1を包む陰線）} \\
\text{Day3}: & C_t < C_{t-1} \quad \text{（下方確定）}
\end{cases}
\quad (-100)
$$

**CDLENGULFING との違い**: 2本足の包み足に3本目の確定足を加えることで、**誤シグナルのリスクを低減**。  
TA-Lib 戻り値: $+100$（買い）/ $-100$（売り）

In [15]:
name = "積水ハウス"
code = "1928.T"

# チャート表示開始日
display_start_date = dt.datetime(2021, 11, 1)
# データ取得終了日(現在の日付を取得)
end_date = dt.datetime(2022, 3, 31)
# データ取得開始日
start_date = display_start_date - dt.timedelta(days=300)

### 積水ハウス (1928.T)
df = get_market_data.get_data_from_yfinance(code, start=start_date.strftime('%Y-%m-%d'), end=end_date.strftime('%Y-%m-%d'))
if isinstance(df.columns, pd.MultiIndex):
    df = df.droplevel(level=1, axis=1)
close = df['Close']

layout = {"height": 600,
          "title": { "text": " {} {} ".format(code, name), "x": 0.5 },
          "xaxis": { "rangeslider": {"visible": False} },
          "yaxis": { "title": "株価 (円)", "side": "left", "tickformat": "," },
          "plot_bgcolor": "light blue",
}

outside = ta.CDL3OUTSIDE(df['Open'], df['High'], df['Low'], df['Close'])
# チャートに表示するテキスト
df["outside_text"] = outside.replace({100: "買い", -100: "売り", 0: ""})
# チャートに表示するマーカーの位置を算出
df["outside_marker"] = (outside / 100 * df["High"]).abs().replace({0: np.nan})

rdf = df[display_start_date:end_date]
# インデクスを文字列型に
rdf.index = pd.to_datetime(rdf.index).strftime('%m-%d-%Y')

# ローソク足
data = [go.Candlestick(x=rdf.index,
                    open=rdf['Open'], high=rdf['High'],
                    low=rdf['Low'], close=rdf['Close'],
                    increasing_line_color='red', 
                    decreasing_line_color='gray',),
        # 検出パターン
        go.Scatter(x=rdf.index,
                y=rdf["outside_marker"],
                mode="markers+text",
                text=rdf["outside_text"],
                textposition="top center",
                name="包み上げ / 包み下げ",
                marker={ "size": 12, "color": "blue",
                        "opacity": 0.6 },
                textfont={ "size": 14, "color": "black" }),
]

fig = go.Figure(layout=go.Layout(layout), data=data)
fig["layout"].update({
    "xaxis": {
        "tickvals": rdf.index[::2],
        "ticktext": [" {} {} ".format(x.split("-")[0], x.split("-")[1]) for x in rdf.index[::2]],
    }
})

fig.show()

[*********************100%***********************]  1 of 1 completed


#### はらみ上げ / はらみ下げ（CDL3INSIDE）— 3本足パターン

はらみ足（2本目で実体が縮小）の後、3本目のローソク足で方向性を**確定**させる反転シグナル。

$$
\text{はらみ上げ（強気反転）}:
\begin{cases}
\text{Day1}: & O_{t-2} > C_{t-2} \quad \text{（陰線）} \\
\text{Day2}: & \min(O_{t-2},C_{t-2}) < \min(O_{t-1},C_{t-1}) \;\text{ かつ }\; \max(O_{t-1},C_{t-1}) < \max(O_{t-2},C_{t-2}) \quad \text{（はらみ足）} \\
\text{Day3}: & C_t > C_{t-1} \quad \text{（上方確定）}
\end{cases}
\quad (+100)
$$

$$
\text{はらみ下げ（弱気反転）}:
\begin{cases}
\text{Day1}: & O_{t-2} < C_{t-2} \quad \text{（陽線）} \\
\text{Day2}: & \text{Day1の実体内に収まる足（はらみ足）} \\
\text{Day3}: & C_t < C_{t-1} \quad \text{（下方確定）}
\end{cases}
\quad (-100)
$$

**CDL3INSIDE vs CDLHARAMI**: CDL3INSIDE は CDLHARAMI の3本目で方向を確定させる、より信頼度の高いシグナル。  
TA-Lib 戻り値: $+100$（買い）/ $-100$（売り）

In [16]:
name = "積水ハウス"
code = "1928.T"

# チャート表示開始日
display_start_date = dt.datetime(2021, 11, 1)
# データ取得終了日(現在の日付を取得)
end_date = dt.datetime(2022, 3, 31)
# データ取得開始日
start_date = display_start_date - dt.timedelta(days=300)

### 積水ハウス (1928.T)
df = get_market_data.get_data_from_yfinance(code, start=start_date.strftime('%Y-%m-%d'), end=end_date.strftime('%Y-%m-%d'))
if isinstance(df.columns, pd.MultiIndex):
    df = df.droplevel(level=1, axis=1)
close = df['Close']

layout = {"height": 600,
          "title": { "text": " {} {} ".format(code, name), "x": 0.5 },
          "xaxis": { "rangeslider": {"visible": False} },
          "yaxis": { "title": "株価 (円)", "side": "left", "tickformat": "," },
          "plot_bgcolor": "light blue",
}

inside = ta.CDL3INSIDE(df['Open'], df['High'], df['Low'], df['Close'])
# チャートに表示するテキスト
df["inside_text"] = inside.replace({100: "買い", -100: "売り", 0: ""})
# チャートに表示するマーカーの位置を算出
df["inside_marker"] = (inside / 100 * df["High"]).abs().replace({0: np.nan})

rdf = df[display_start_date:end_date]
# インデクスを文字列型に
rdf.index = pd.to_datetime(rdf.index).strftime('%m-%d-%Y')

# ローソク足
data = [go.Candlestick(x=rdf.index,
                    open=rdf['Open'], high=rdf['High'],
                    low=rdf['Low'], close=rdf['Close'],
                    increasing_line_color='red', 
                    decreasing_line_color='gray',),
        # 検出パターン
        go.Scatter(x=rdf.index,
                y=rdf["inside_marker"],
                mode="markers+text",
                text=rdf["inside_text"],
                textposition="top center",
                name="はらみ上げ / はらみ下げ",
                marker={ "size": 12, "color": "blue",
                        "opacity": 0.6 },
                textfont={ "size": 14, "color": "black" }),
]

fig = go.Figure(layout=go.Layout(layout), data=data)
fig["layout"].update({
    "xaxis": {
        "tickvals": rdf.index[::2],
        "ticktext": [" {} {} ".format(x.split("-")[0], x.split("-")[1]) for x in rdf.index[::2]],
    }
})

fig.show()

[*********************100%***********************]  1 of 1 completed
